> Resetto le variabili.

In [1]:
%reset

Once deleted, variables cannot be recovered. Proceed (y/[n])? n
Nothing done.


> Importo moduli.



In [2]:
import os
import glob
import nibabel
import numpy as np
import matplotlib.pyplot as plt
from sklearn import svm
from sklearn.decomposition import PCA
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler
from google.colab import drive
drive.mount('/content/drive',force_remount=True)

Mounted at /content/drive


# PRE-PROCESSING

> Variabili utili.

In [3]:
image_size = np.array([61,73,61])

> Creo matrice dei pazienti.

In [4]:
os.chdir('/content/drive/My Drive/Brain - Tatiana/fALFF/4') 
images_pazienti = glob.glob('*.nii', recursive=True)
num_pazienti = len(images_pazienti) 

X_pazienti = np.zeros((num_pazienti,np.product(image_size)))
t = 0
for fMRI in images_pazienti:
  file_nii = nibabel.load(fMRI)
  img = np.array(file_nii.dataobj)
  X_pazienti[t,:] = np.reshape(img,(1,np.product(image_size)))
  t = t + 1

y_pazienti = np.ones((num_pazienti,1))

> Creao matrice dei controlli.

In [5]:
os.chdir('/content/drive/My Drive/Brain - Tatiana/fALFF/5') 
images_controlli = glob.glob('*.nii', recursive=True) 
num_controlli = len(images_controlli)

X_controlli = np.zeros((num_controlli,np.product(image_size)))
t = 0
for fMRI in images_controlli:
  file_nii = nibabel.load(fMRI)
  img = np.array(file_nii.dataobj)
  X_controlli[t,:] = np.reshape(img,(1,np.product(image_size)))
  t = t + 1

y_controlli = np.zeros((num_controlli,1))

> Unisco le matrici.

In [6]:
X = np.concatenate((X_pazienti,X_controlli))
y = np.concatenate((y_pazienti,y_controlli))

Elimino dalla matrice X le colonne con somma 0 (i.e. le colonne corrispondenti a voxel neri in tutti i soggetti)

In [7]:
initial_num_cols = X.shape[1]

mask = (X == 0).all(0)
column_indices = np.where(mask)[0]
X = X[:,~mask]

final_num_cols = X.shape[1]

print(str(initial_num_cols - final_num_cols) + ' columns were dropped from the dataset')


200802 columns were dropped from the dataset


# PCA

In [8]:


scaler1 = StandardScaler()
scaler1.fit(X)
X_scaled = scaler1.transform(X)

pca = PCA(n_components=25)
pca.fit(X_scaled)
X_pca = pca.transform(X_scaled)
#X_pca = X_scaled

In [41]:
from sklearn.decomposition import FastICA
transformer = FastICA(n_components=400, random_state=0, fun='cube',whiten=True,algorithm='deflation')
X_ica = transformer.fit_transform(X_scaled)
X_pca=X_ica

/usr/local/lib/python3.7/dist-packages/sklearn/decomposition/_fastica.py:470: UserWarning: n_components is too large: it will be set to 34
  % n_components


In [9]:
X.shape

(34, 70831)

# Parameter estimation

In [42]:
# Set the parameters by cross-validation
tuned_parameters = [{'kernel': ['rbf','linear','sigmoid','poly'], 'gamma': [1e-3, 1e-4],
                     'C': [1, 10, 100, 1000]}]

clf = GridSearchCV(svm.SVC(), tuned_parameters, scoring='accuracy', cv=LeaveOneOut())
clf.fit(X_pca, y.ravel())

print("Best parameters set found on development set:")
print()
print(clf.best_params_)
print()
print("Grid scores on development set:")
print()
means = clf.cv_results_['mean_test_score']
stds = clf.cv_results_['std_test_score']
for mean, std, params in zip(means, stds, clf.cv_results_['params']):
        print("%0.3f (+/-%0.03f) for %r"
              % (mean, std * 2, params))

Best parameters set found on development set:

{'C': 1, 'gamma': 0.001, 'kernel': 'rbf'}

Grid scores on development set:

0.000 (+/-0.000) for {'C': 1, 'gamma': 0.001, 'kernel': 'rbf'}
0.000 (+/-0.000) for {'C': 1, 'gamma': 0.001, 'kernel': 'linear'}
0.000 (+/-0.000) for {'C': 1, 'gamma': 0.001, 'kernel': 'sigmoid'}
0.000 (+/-0.000) for {'C': 1, 'gamma': 0.001, 'kernel': 'poly'}
0.000 (+/-0.000) for {'C': 1, 'gamma': 0.0001, 'kernel': 'rbf'}
0.000 (+/-0.000) for {'C': 1, 'gamma': 0.0001, 'kernel': 'linear'}
0.000 (+/-0.000) for {'C': 1, 'gamma': 0.0001, 'kernel': 'sigmoid'}
0.000 (+/-0.000) for {'C': 1, 'gamma': 0.0001, 'kernel': 'poly'}
0.000 (+/-0.000) for {'C': 10, 'gamma': 0.001, 'kernel': 'rbf'}
0.000 (+/-0.000) for {'C': 10, 'gamma': 0.001, 'kernel': 'linear'}
0.000 (+/-0.000) for {'C': 10, 'gamma': 0.001, 'kernel': 'sigmoid'}
0.000 (+/-0.000) for {'C': 10, 'gamma': 0.001, 'kernel': 'poly'}
0.000 (+/-0.000) for {'C': 10, 'gamma': 0.0001, 'kernel': 'rbf'}
0.000 (+/-0.000) for {'C

# Leave-one out

In [44]:
# evaluate model
best_clf = clf.best_estimator_
scores = cross_val_score(best_clf, X_pca, y, scoring='accuracy', cv=LeaveOneOut(), n_jobs=-1)
# report performance
print('Accuracy: %.3f (%.3f)' % (np.mean(scores), np.std(scores)))

Accuracy: 0.000 (0.000)


In [45]:
from sklearn.model_selection import StratifiedShuffleSplit

sss = StratifiedShuffleSplit(n_splits=1000, test_size=0.1, random_state=0)
sss.get_n_splits(X_pca, y.ravel())

1000

In [46]:
best_clf = clf.best_estimator_
scores = cross_val_score(best_clf, X_pca, y.ravel(), scoring='accuracy', cv=sss, n_jobs=-1)
# report performance
print('Accuracy: %.3f (%.3f)' % (np.mean(scores), np.std(scores)))

Accuracy: 0.485 (0.191)


## CV optimization

In [14]:
# Set the parameters by cross-validation
tuned_parameters = [{'kernel': ['rbf','linear','sigmoid','poly'], 'gamma': [1e-2,1e-3, 1e-4],
                     'C': [.1,1, 10, 100, 1000]}]

clf1 = GridSearchCV(svm.SVC(), tuned_parameters, scoring='accuracy',cv=10)
clf1.fit(X_pca, y.ravel())

print("Best parameters set found on development set:")
print()
print(clf1.best_params_)
print()
print("Grid scores on development set:")
print()
means = clf1.cv_results_['mean_test_score']
stds = clf1.cv_results_['std_test_score']
for mean, std, params in zip(means, stds, clf1.cv_results_['params']):
        print("%0.3f (+/-%0.03f) for %r"
              % (mean, std * 2, params))

Best parameters set found on development set:

{'C': 0.1, 'gamma': 0.0001, 'kernel': 'poly'}

Grid scores on development set:

0.425 (+/-0.263) for {'C': 0.1, 'gamma': 0.01, 'kernel': 'rbf'}
0.408 (+/-0.444) for {'C': 0.1, 'gamma': 0.01, 'kernel': 'linear'}
0.292 (+/-0.327) for {'C': 0.1, 'gamma': 0.01, 'kernel': 'sigmoid'}
0.400 (+/-0.464) for {'C': 0.1, 'gamma': 0.01, 'kernel': 'poly'}
0.400 (+/-0.163) for {'C': 0.1, 'gamma': 0.001, 'kernel': 'rbf'}
0.408 (+/-0.444) for {'C': 0.1, 'gamma': 0.001, 'kernel': 'linear'}
0.258 (+/-0.369) for {'C': 0.1, 'gamma': 0.001, 'kernel': 'sigmoid'}
0.400 (+/-0.464) for {'C': 0.1, 'gamma': 0.001, 'kernel': 'poly'}
0.425 (+/-0.263) for {'C': 0.1, 'gamma': 0.0001, 'kernel': 'rbf'}
0.408 (+/-0.444) for {'C': 0.1, 'gamma': 0.0001, 'kernel': 'linear'}
0.375 (+/-0.479) for {'C': 0.1, 'gamma': 0.0001, 'kernel': 'sigmoid'}
0.467 (+/-0.249) for {'C': 0.1, 'gamma': 0.0001, 'kernel': 'poly'}
0.425 (+/-0.263) for {'C': 1, 'gamma': 0.01, 'kernel': 'rbf'}
0.408 (

In [28]:
from sklearn.model_selection import StratifiedShuffleSplit

sss = StratifiedShuffleSplit(n_splits=1000, test_size=.1, random_state=0)
sss.get_n_splits(X_pca, y.ravel())
print(sss)

StratifiedShuffleSplit(n_splits=1000, random_state=0, test_size=0.1,
            train_size=None)


In [16]:
best_clf1 = clf1.best_estimator_
scores = cross_val_score(best_clf1, X_pca, y.ravel(), scoring='accuracy', cv=sss, n_jobs=-1)
# report performance
print('Accuracy: %.3f (%.3f)' % (np.mean(scores), np.std(scores)))

Accuracy: 0.474 (0.092)


In [21]:
# sometimes you have to run this cell twice on colab
!apt-get install swig -y
!pip install Cython numpy
!pip install scikit-learn --upgrade
!pip install auto-sklearn --upgrade
!pip install dask distributed --upgrade
!pip install pipelineprofiler

Reading package lists... Done
Building dependency tree       
Reading state information... Done
swig is already the newest version (3.0.12-1).
The following package was automatically installed and is no longer required:
  libnvidia-common-460
Use 'apt autoremove' to remove it.
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.
     |████████████████████████████████| 23.2 MB 30.8 MB/s 
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 0.24.2
    Uninstalling scikit-learn-0.24.2:
      Successfully uninstalled scikit-learn-0.24.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
auto-sklearn 0.14.0 requires scikit-learn<0.25.0,>=0.24.0, but you have scikit-learn 1.0.1 which is incompatible.


  Using cached scikit_learn-0.24.2-cp37-cp37m-manylinux2010_x86_64.whl (22.3 MB)
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.0.1
    Uninstalling scikit-learn-1.0.1:
      Successfully uninstalled scikit-learn-1.0.1


     |████████████████████████████████| 1.0 MB 31.7 MB/s 
     |████████████████████████████████| 791 kB 49.5 MB/s 
  Attempting uninstall: dask
    Found existing installation: dask 2021.6.2
    Uninstalling dask-2021.6.2:
      Successfully uninstalled dask-2021.6.2
  Attempting uninstall: distributed
    Found existing installation: distributed 2021.6.2
    Uninstalling distributed-2021.6.2:
      Successfully uninstalled distributed-2021.6.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
auto-sklearn 0.14.0 requires dask<2021.07, but you have dask 2021.10.0 which is incompatible.
auto-sklearn 0.14.0 requires distributed<2021.07,>=2.2.0, but you have distributed 2021.10.0 which is incompatible.


     |████████████████████████████████| 880 kB 35.8 MB/s 


In [22]:
import autosklearn.classification
import sklearn.model_selection
import sklearn.datasets
import sklearn.metrics


In [30]:

automl = autosklearn.classification.AutoSklearnClassifier(time_left_for_this_task=60*25) #Auto-sklearn searches pipelines for 5 minutes

automl.fit(X, y)

[WARNING] [2021-11-02 10:31:27,392:Client-EnsembleBuilder] No models better than random - using Dummy loss!Number of models besides current dummy model: 1. Number of dummy models: 1
[WARNING] [2021-11-02 10:33:58,508:Client-EnsembleBuilder] No models better than random - using Dummy loss!Number of models besides current dummy model: 1. Number of dummy models: 1
[WARNING] [2021-11-02 10:34:08,823:Client-EnsembleBuilder] No models better than random - using Dummy loss!Number of models besides current dummy model: 2. Number of dummy models: 1
[WARNING] [2021-11-02 10:34:27,763:Client-EnsembleBuilder] No models better than random - using Dummy loss!Number of models besides current dummy model: 3. Number of dummy models: 1
[WARNING] [2021-11-02 10:34:46,082:Client-EnsembleBuilder] No models better than random - using Dummy loss!Number of models besides current dummy model: 4. Number of dummy models: 1
[WARNING] [2021-11-02 10:34:56,123:Client-EnsembleBuilder] No models better than random - 

AutoSklearnClassifier(per_run_time_limit=150, time_left_for_this_task=1500)

In [31]:
import PipelineProfiler

data = PipelineProfiler.import_autosklearn(automl)
PipelineProfiler.plot_pipeline_matrix(data)

Output hidden; open in https://colab.research.google.com to view.

In [35]:
# show all models
show_modes_str=automl.show_models()
sprint_statistics_str = automl.sprint_statistics()
print(show_modes_str)
print(sprint_statistics_str)

[(0.320000, SimpleClassificationPipeline({'balancing:strategy': 'weighting', 'classifier:__choice__': 'random_forest', 'data_preprocessor:__choice__': 'feature_type', 'feature_preprocessor:__choice__': 'extra_trees_preproc_for_classification', 'classifier:random_forest:bootstrap': 'True', 'classifier:random_forest:criterion': 'gini', 'classifier:random_forest:max_depth': 'None', 'classifier:random_forest:max_features': 0.43999367631975456, 'classifier:random_forest:max_leaf_nodes': 'None', 'classifier:random_forest:min_impurity_decrease': 0.0, 'classifier:random_forest:min_samples_leaf': 2, 'classifier:random_forest:min_samples_split': 2, 'classifier:random_forest:min_weight_fraction_leaf': 0.0, 'data_preprocessor:feature_type:categorical_transformer:categorical_encoding:__choice__': 'no_encoding', 'data_preprocessor:feature_type:categorical_transformer:category_coalescence:__choice__': 'no_coalescense', 'data_preprocessor:feature_type:numerical_transformer:imputation:strategy': 'most_

In [ ]:

automl2 = autosklearn.classification.AutoSklearnClassifier(time_left_for_this_task=60*25) #Auto-sklearn searches pipelines for 5 minutes

automl2.fit(X_pca, y)

[WARNING] [2021-11-02 13:22:51,630:Client-EnsembleBuilder] No models better than random - using Dummy loss!Number of models besides current dummy model: 1. Number of dummy models: 1
[WARNING] [2021-11-02 13:22:53,896:Client-EnsembleBuilder] No models better than random - using Dummy loss!Number of models besides current dummy model: 2. Number of dummy models: 1
[WARNING] [2021-11-02 13:22:55,115:Client-EnsembleBuilder] No models better than random - using Dummy loss!Number of models besides current dummy model: 2. Number of dummy models: 1


AutoSklearnClassifier(per_run_time_limit=150, time_left_for_this_task=1500)

In [ ]:
# show all models
show_modes_str2=automl2.show_models()
sprint_statistics_str2 = automl2.sprint_statistics()
print(show_modes_str2)
print(sprint_statistics_str2)

In [48]:
best_clf1 = automl
sss = StratifiedShuffleSplit(n_splits=1000, test_size=.1, random_state=0)
sss.get_n_splits(X, y.ravel())
print(sss)


StratifiedShuffleSplit(n_splits=1000, random_state=0, test_size=0.1,
            train_size=None)


In [49]:
scores = cross_val_score(best_clf1, X, y.ravel(), scoring='accuracy', cv=sss, n_jobs=-1)
# report performance
print('Accuracy: %.3f (%.3f)' % (np.mean(scores), np.std(scores)))

[ERROR] [2021-11-02 11:27:56,237:concurrent.futures] exception calling callback for <Future at 0x7fe143f7a950 state=finished raised BrokenProcessPool>
joblib.externals.loky.process_executor._RemoteTraceback: 
"""
Traceback (most recent call last):
  File "/usr/local/lib/python3.7/dist-packages/joblib/externals/loky/process_executor.py", line 404, in _process_worker
    call_item = call_queue.get(block=True, timeout=timeout)
  File "/usr/lib/python3.7/multiprocessing/queues.py", line 113, in get
    return _ForkingPickler.loads(res)
  File "/usr/local/lib/python3.7/dist-packages/autosklearn/__init__.py", line 13, in <module>
    dependencies.verify_packages(requirements)
  File "/usr/local/lib/python3.7/dist-packages/autosklearn/util/dependencies.py", line 25, in verify_packages
    _verify_package(name, operation, version)
  File "/usr/local/lib/python3.7/dist-packages/autosklearn/util/dependencies.py", line 62, in _verify_package
    required_version)
autosklearn.util.dependencies.Inc

BrokenProcessPool: ignored

In [52]:
from autosklearn.experimental.askl2 import AutoSklearn2Classifier

automlclassifierV2 = AutoSklearn2Classifier(time_left_for_this_task=25*60, per_run_time_limit=60)
automlclassifierV2.fit(X, y)

/usr/local/lib/python3.7/dist-packages/smac/intensification/parallel_scheduling.py:155: UserWarning: SuccessiveHalving is intended to be used with more than 1 worker but num_workers=1
  num_workers


[WARNING] [2021-11-02 11:47:30,024:Client-EnsembleBuilder] No models better than random - using Dummy loss!Number of models besides current dummy model: 1. Number of dummy models: 1
[WARNING] [2021-11-02 11:47:59,569:Client-EnsembleBuilder] No models better than random - using Dummy loss!Number of models besides current dummy model: 2. Number of dummy models: 1
[WARNING] [2021-11-02 11:48:10,351:Client-EnsembleBuilder] No models better than random - using Dummy loss!Number of models besides current dummy model: 3. Number of dummy models: 1
[WARNING] [2021-11-02 11:49:11,526:Client-EnsembleBuilder] No models better than random - using Dummy loss!Number of models besides current dummy model: 3. Number of dummy models: 1
[WARNING] [2021-11-02 11:49:25,637:Client-EnsembleBuilder] No models better than random - using Dummy loss!Number of models besides current dummy model: 3. Number of dummy models: 1
[WARNING] [2021-11-02 11:50:26,784:Client-EnsembleBuilder] No models better than random - 

/usr/local/lib/python3.7/dist-packages/smac/intensification/parallel_scheduling.py:155: UserWarning: SuccessiveHalving is intended to be used with more than 1 worker but num_workers=1
  num_workers
/usr/local/lib/python3.7/dist-packages/smac/intensification/parallel_scheduling.py:155: UserWarning: SuccessiveHalving is intended to be used with more than 1 worker but num_workers=1
  num_workers
/usr/local/lib/python3.7/dist-packages/smac/intensification/parallel_scheduling.py:155: UserWarning: SuccessiveHalving is intended to be used with more than 1 worker but num_workers=1
  num_workers
/usr/local/lib/python3.7/dist-packages/smac/intensification/parallel_scheduling.py:155: UserWarning: SuccessiveHalving is intended to be used with more than 1 worker but num_workers=1
  num_workers
/usr/local/lib/python3.7/dist-packages/smac/intensification/parallel_scheduling.py:155: UserWarning: SuccessiveHalving is intended to be used with more than 1 worker but num_workers=1
  num_workers
/usr/local

AutoSklearn2Classifier(metric=accuracy, per_run_time_limit=60,
                       time_left_for_this_task=1500)

In [51]:
import PipelineProfiler

data = PipelineProfiler.import_autosklearn(automlclassifierV2)
PipelineProfiler.plot_pipeline_matrix(data)

Output hidden; open in https://colab.research.google.com to view.